In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import requests
import os
import pandas as pd
import numpy as np
import sys
from pathlib import Path
sys.path.append(os.path.abspath(".."))
import src.raw_preprocessing as rp
import src.feature_engineering as fe
import src.create_dataset as cd
from datetime import date, datetime, timedelta
from vacances_scolaires_france import SchoolHolidayDates

import openmeteo_requests
import requests_cache
from retry_requests import retry

In [70]:
df = pd.read_parquet("../data/final_datasets/datasets_linear_models/conso_v3_linear.parquet")

In [71]:
df.head()

,Consommation,Zone_A,Zone_B,Zone_C,Vacances de la Toussaint,Vacances de Noël,Vacances d'Hiver,Vacances de Printemps,Vacances d'Été,public_holidays,...,is_weekend_x_season_Summer,is_holiday_x_season_Summer,hour_x_season_Winter,is_weekend_x_season_Winter,is_holiday_x_season_Winter,temp_x_humidity,temp_x_wind,humidity_x_wind,HDD,CDD
336,76986.0,0,0,0,0,0,0,0,0,0,...,0,0,0.0,0,0,-23.625160,-0.448757,158.664136,18.258496,0.0
337,75624.0,0,0,0,0,0,0,0,0,0,...,0,0,0.5,0,0,-51.046198,-1.881718,307.182089,18.559192,0.0
338,73317.0,0,0,0,0,0,0,0,0,0,...,0,0,1.0,0,0,-78.401620,-1.624052,172.203081,18.859888,0.0
339,74096.0,0,0,0,0,0,0,0,0,0,...,0,0,1.5,0,0,-60.000035,-2.217065,306.450364,18.658847,0.0
340,73854.0,0,0,0,0,0,0,0,0,0,...,0,0,2.0,0,0,-41.642012,-0.884911,175.819505,18.457807,0.0


In [72]:
df.columns

Index(['Consommation', 'Zone_A', 'Zone_B', 'Zone_C',
       'Vacances de la Toussaint', 'Vacances de Noël', 'Vacances d'Hiver',
       'Vacances de Printemps', 'Vacances d'Été', 'public_holidays', '44T',
       '69T', '59T', '75T', '13T', '33T', 'T', 'U', 'FF', 'PMER', 'RR1',
       'year', 'month', 'hour', 'day_of_week', 'is_weekend', 'hour_sin',
       'hour_cos', 'day_of_week_sin', 'day_of_week_cos', 'month_sin',
       'month_cos', 'lagged_1', 'lagged_2', 'lagged_48', 'lagged_336',
       'rolling_mean_24h', 'rolling_std_24h', 'rolling_mean_7d',
       'rolling_std_7d', 'rolling_max_24h', 'rolling_min_24h',
       'consumption_diff_1', 'consumption_diff_48', 'consumption_pct_change_1',
       'consumption_pct_change_48', 'season_Spring', 'season_Summer',
       'season_Winter', 'temp_sq', 'humidity_sq', 'hour_x_is_weekend',
       'hour_x_is_holiday', 'hour_x_dow', 'hour_x_month', 'is_weekend_x_month',
       'is_holiday_x_month', 'hour_x_temp', 'hour_x_humidity', 'hour_x_wind',
  

In [91]:
cd.download_monthly_data()

conso_energie_2026.zip already exists


In [5]:
df = rp.conso_preprocess(Path("../data/conso/real_time_conso/"))

df = df[df["Heures"].apply(lambda x: x.minute in {00, 30})]

df = df.reset_index(drop=True)
df.loc[len(df)] = None

In [6]:
df = fe.lagged_consumption(df)

In [7]:
df

,Date,Heures,Consommation,lagged_1,lagged_2,lagged_48,lagged_336
0,2026-05-01,00:00:00,41688.0,NaN,NaN,NaN,NaN
1,2026-05-01,00:30:00,40393.0,41688.0,NaN,NaN,NaN
2,2026-05-01,01:00:00,38108.0,40393.0,41688.0,NaN,NaN
3,2026-05-01,01:30:00,37744.0,38108.0,40393.0,NaN,NaN
4,2026-05-01,02:00:00,36708.0,37744.0,38108.0,NaN,NaN
...,...,...,...,...,...,...,...
3818,2026-07-19,13:00:00,42182.0,42953.0,41863.0,46948.0,45512.0
3819,2026-07-19,13:30:00,41187.0,42182.0,42953.0,46739.0,44682.0
3820,2026-07-19,14:00:00,41283.0,41187.0,42182.0,46889.0,44856.0
3821,2026-07-19,14:30:00,41408.0,41283.0,41187.0,46745.0,45452.0


In [8]:
t = df["Heures"].iloc[-2]

new_time = (
    datetime.combine(datetime.today(), t)
    + timedelta(minutes=30)
).time()

In [9]:
df.loc[len(df)-1, "Heures"] = new_time

In [10]:
df.loc[len(df)-1, "Date"] = datetime.today().strftime('%Y-%m-%d')

In [11]:
df

,Date,Heures,Consommation,lagged_1,lagged_2,lagged_48,lagged_336
0,2026-05-01,00:00:00,41688.0,NaN,NaN,NaN,NaN
1,2026-05-01,00:30:00,40393.0,41688.0,NaN,NaN,NaN
2,2026-05-01,01:00:00,38108.0,40393.0,41688.0,NaN,NaN
3,2026-05-01,01:30:00,37744.0,38108.0,40393.0,NaN,NaN
4,2026-05-01,02:00:00,36708.0,37744.0,38108.0,NaN,NaN
...,...,...,...,...,...,...,...
3818,2026-07-19,13:00:00,42182.0,42953.0,41863.0,46948.0,45512.0
3819,2026-07-19,13:30:00,41187.0,42182.0,42953.0,46739.0,44682.0
3820,2026-07-19,14:00:00,41283.0,41187.0,42182.0,46889.0,44856.0
3821,2026-07-19,14:30:00,41408.0,41283.0,41187.0,46745.0,45452.0


#### Adding the name of the holidays

In [101]:
from pprint import pprint
today = date(2026, 5, 16).isoformat()

url = "https://data.education.gouv.fr/api/explore/v2.1/catalog/datasets/fr-en-calendrier-scolaire/records"

params = {
    "where": f"start_date <= date'{today}' AND end_date >= date'{today}' AND zones = 'Zone C'"
}

response = requests.get(url, params=params)

In [12]:
#Adding the infos about the holidays
today = date.today().isoformat()
url = "https://data.education.gouv.fr/api/explore/v2.1/catalog/datasets/fr-en-calendrier-scolaire/records"

zones = ['A', 'B', 'C']
vacances = ['vacances de la toussaint', 'vacances de noël', "vacances d'hiver",'vacances de printemps', "vacances d'été"]
feries = [
    "jour de l'an",
    "lundi de pâques",
    "fête du travail",
    "victoire 1945",
    "ascension",
    "lundi de pentecôte",
    "fête nationale",
    "assomption",
    "toussaint",
    "armistice",
    "noël",
    "pont de l'ascension",
]

#finished_with_holidays = False
df.loc[:, "public_holidays"] = 0
df.loc[:, "Vacances de la Toussaint"] = 0
df.loc[:, "Vacances de Noël"] = 0
df.loc[:, "Vacances d'Hiver"] = 0
df.loc[:, "Vacances de Printemps"] = 0
df.loc[:, "Vacances d'Été"] = 0

for z in zones:
    params = {
        "where": f"start_date <= date'{today}' AND end_date >= date'{today}' AND zones = 'Zone {z}'"
    }
    response = requests.get(url, params=params).json()
    # If no holidays we set the columns with the value 0
    if response["total_count"] == 0:
        continue

    else : 
        
        for event in response["results"]:
            # We check if it's school holidays
            if event["description"].lower() in vacances:
                if event["description"].lower() == "vacances de la toussaint":
                    df.loc[:, "Vacances de la Toussaint"] = 1
                    
                elif event["description"].lower() == "vacances de noël":
                    df.loc[:, "Vacances de Noël"] = 1
                    
                elif event["description"].lower() == "vacances d'hiver":
                    df.loc[:, "Vacances d'Hiver"] = 1
                    
                elif event["description"].lower() == "vacances de printemps":
                    df.loc[:, "Vacances de Printemps"] = 1

                elif event["description"].lower() == "vacances d'été":
                    df.loc[:, "Vacances d'Été"] = 1
            # Or if it's public holidays (jours fériés)
            elif event in feries:
                df.loc[:, "public_holidays"] = 1

In [13]:
df

,Date,Heures,Consommation,lagged_1,lagged_2,lagged_48,lagged_336,public_holidays,Vacances de la Toussaint,Vacances de Noël,Vacances d'Hiver,Vacances de Printemps,Vacances d'Été
0,2026-05-01,00:00:00,41688.0,NaN,NaN,NaN,NaN,0,0,0,0,0,1
1,2026-05-01,00:30:00,40393.0,41688.0,NaN,NaN,NaN,0,0,0,0,0,1
2,2026-05-01,01:00:00,38108.0,40393.0,41688.0,NaN,NaN,0,0,0,0,0,1
3,2026-05-01,01:30:00,37744.0,38108.0,40393.0,NaN,NaN,0,0,0,0,0,1
4,2026-05-01,02:00:00,36708.0,37744.0,38108.0,NaN,NaN,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3818,2026-07-19,13:00:00,42182.0,42953.0,41863.0,46948.0,45512.0,0,0,0,0,0,1
3819,2026-07-19,13:30:00,41187.0,42182.0,42953.0,46739.0,44682.0,0,0,0,0,0,1
3820,2026-07-19,14:00:00,41283.0,41187.0,42182.0,46889.0,44856.0,0,0,0,0,0,1
3821,2026-07-19,14:30:00,41408.0,41283.0,41187.0,46745.0,45452.0,0,0,0,0,0,1


In [14]:
df = fe.date_and_hour_pred(df)
df = fe.cyclical_encoding(df)
df = fe.rolling_window(df)
df = fe.lagged_trend(df)

In [15]:
row = df.iloc[-1]
row

Consommation                                 NaN
lagged_1                                 41408.0
lagged_2                                 41283.0
lagged_48                                46359.0
lagged_336                               44998.0
public_holidays                                0
Vacances de la Toussaint                       0
Vacances de Noël                               0
Vacances d'Hiver                               0
Vacances de Printemps                          0
Vacances d'Été                                 1
full_date                    2026-07-30 15:00:00
year                                        2026
month                                          7
hour                                        15.0
day_of_week                                    3
is_weekend                                     0
hour_sin                               -0.707107
hour_cos                               -0.707107
day_of_week_sin                         0.433884
day_of_week_cos     

In [16]:
pred = pd.DataFrame([row] * 10)

In [17]:
pred["full_date"] = row["full_date"] + pd.to_timedelta(range(10), unit="m") * 30

In [18]:
pred["full_date"].iloc[-1]

Timestamp('2026-07-30 19:30:00')

In [19]:
pred

,Consommation,lagged_1,lagged_2,lagged_48,lagged_336,public_holidays,Vacances de la Toussaint,Vacances de Noël,Vacances d'Hiver,Vacances de Printemps,...,rolling_mean_24h,rolling_std_24h,rolling_mean_7d,rolling_std_7d,rolling_max_24h,rolling_min_24h,consumption_diff_1,consumption_diff_48,consumption_pct_change_1,consumption_pct_change_48
3822,NaN,41408.0,41283.0,46359.0,44998.0,0,0,0,0,0,...,40340.479167,4656.129417,45952.77381,5955.083204,46374.0,33277.0,125.0,-5337.0,0.003028,-0.114173
3822,NaN,41408.0,41283.0,46359.0,44998.0,0,0,0,0,0,...,40340.479167,4656.129417,45952.77381,5955.083204,46374.0,33277.0,125.0,-5337.0,0.003028,-0.114173
3822,NaN,41408.0,41283.0,46359.0,44998.0,0,0,0,0,0,...,40340.479167,4656.129417,45952.77381,5955.083204,46374.0,33277.0,125.0,-5337.0,0.003028,-0.114173
3822,NaN,41408.0,41283.0,46359.0,44998.0,0,0,0,0,0,...,40340.479167,4656.129417,45952.77381,5955.083204,46374.0,33277.0,125.0,-5337.0,0.003028,-0.114173
3822,NaN,41408.0,41283.0,46359.0,44998.0,0,0,0,0,0,...,40340.479167,4656.129417,45952.77381,5955.083204,46374.0,33277.0,125.0,-5337.0,0.003028,-0.114173
3822,NaN,41408.0,41283.0,46359.0,44998.0,0,0,0,0,0,...,40340.479167,4656.129417,45952.77381,5955.083204,46374.0,33277.0,125.0,-5337.0,0.003028,-0.114173
3822,NaN,41408.0,41283.0,46359.0,44998.0,0,0,0,0,0,...,40340.479167,4656.129417,45952.77381,5955.083204,46374.0,33277.0,125.0,-5337.0,0.003028,-0.114173
3822,NaN,41408.0,41283.0,46359.0,44998.0,0,0,0,0,0,...,40340.479167,4656.129417,45952.77381,5955.083204,46374.0,33277.0,125.0,-5337.0,0.003028,-0.114173
3822,NaN,41408.0,41283.0,46359.0,44998.0,0,0,0,0,0,...,40340.479167,4656.129417,45952.77381,5955.083204,46374.0,33277.0,125.0,-5337.0,0.003028,-0.114173
3822,NaN,41408.0,41283.0,46359.0,44998.0,0,0,0,0,0,...,40340.479167,4656.129417,45952.77381,5955.083204,46374.0,33277.0,125.0,-5337.0,0.003028,-0.114173


#### We'll add the temperature for each hour and interpolate the values for the hours with this format : hh:30

### RTE France API

In [86]:
id_client = "863fa354-33b4-4bf6-a844-3dd668062f92"
id_secret = "83b2284d-b40f-4627-916f-d35473030cbf"
url = "https://digital.iservices.rte-france.com/token/oauth/"

response = requests.post(url, auth=(id_client, id_secret))

In [87]:
response.json()

{'access_token': 'k474YPtYGg8Im6fzkvcoIUeWaPI4fzxRjj6ankSZK5EghTijE48cHG',
 'token_type': 'Bearer',
 'expires_in': 3600}

In [88]:
token = response.json()["access_token"]
headers = {
    "Authorization" : f"Bearer {token}"
}

url = "https://digital.iservices.rte-france.com/open_api/consumption/v1/short_term"

data = requests.get(url, headers=headers)

In [89]:
data.json()

{'short_term': [{'type': 'REALISED',
   'start_date': '2026-07-17T00:00:00+02:00',
   'end_date': '2026-07-18T00:00:00+02:00',
   'values': [{'start_date': '2026-07-17T00:00:00+02:00',
     'end_date': '2026-07-17T00:15:00+02:00',
     'updated_date': '2026-07-17T13:05:47+02:00',
     'value': 47387},
    {'start_date': '2026-07-17T00:15:00+02:00',
     'end_date': '2026-07-17T00:30:00+02:00',
     'updated_date': '2026-07-17T13:05:47+02:00',
     'value': 46988},
    {'start_date': '2026-07-17T00:30:00+02:00',
     'end_date': '2026-07-17T00:45:00+02:00',
     'updated_date': '2026-07-17T13:05:48+02:00',
     'value': 45765},
    {'start_date': '2026-07-17T00:45:00+02:00',
     'end_date': '2026-07-17T01:00:00+02:00',
     'updated_date': '2026-07-17T13:05:48+02:00',
     'value': 44762},
    {'start_date': '2026-07-17T01:00:00+02:00',
     'end_date': '2026-07-17T01:15:00+02:00',
     'updated_date': '2026-07-17T13:05:49+02:00',
     'value': 43709},
    {'start_date': '2026-07-17T01

### Open-Meteo API

In [21]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"
lat_long = {
    '13': {'ville': 'Marseille', 'latitude': 43.2965, 'longitude': 5.3698},     # Bouches-du-Rhône
    '33': {'ville': 'Bordeaux',  'latitude': 44.8378, 'longitude': -0.5792},    # Gironde
    '44': {'ville': 'Nantes',    'latitude': 47.2184, 'longitude': -1.5536},    # Loire-Atlantique
    '59': {'ville': 'Lille',     'latitude': 50.6292, 'longitude': 3.0573},     # Nord
    '69': {'ville': 'Lyon',      'latitude': 45.7640, 'longitude': 4.8357},     # Rhône
    '75': {'ville': 'Paris',     'latitude': 48.8566, 'longitude': 2.3522},     # Paris
}

for k, v in lat_long.items():
    
    params = {
    	"latitude": v["latitude"],
    	"longitude": v["longitude"],
    	"hourly": ["temperature_2m", "relative_humidity_2m", "rain", "surface_pressure", "wind_speed_10m"],
    	"timezone": "Europe/London",
    	"past_days": 7,
    	"forecast_days": 1,
    }
    responses = openmeteo.weather_api(url, params = params)
    
    # Process first location. Add a for-loop for multiple locations or weather models
    response = responses[0]
    print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
    print(f"Elevation: {response.Elevation()} m asl")
    print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
    print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")
    
    # Process hourly data. The order of variables needs to be the same as requested.
    hourly = response.Hourly()
    hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
    hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
    hourly_rain = hourly.Variables(2).ValuesAsNumpy()
    hourly_surface_pressure = hourly.Variables(3).ValuesAsNumpy()
    hourly_wind_speed_10m = hourly.Variables(4).ValuesAsNumpy()
    
    hourly_data = {
    	"date": pd.date_range(
    		start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
    		end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
    		freq = pd.Timedelta(seconds = hourly.Interval()),
    		inclusive = "left"
    	).tz_convert(response.Timezone().decode())
    }
    
    hourly_data["temperature_2m"] = hourly_temperature_2m
    hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
    hourly_data["rain"] = hourly_rain
    hourly_data["surface_pressure"] = hourly_surface_pressure
    hourly_data["wind_speed_10m"] = hourly_wind_speed_10m
    
    hourly_dataframe = pd.DataFrame(data = hourly_data)
    print("\nHourly data\n", hourly_dataframe)source venv/bin/activate


Coordinates: 43.29999923706055°N 5.369998931884766°E
Elevation: 6.0 m asl
Timezone: b'Europe/London'b'GMT+1'
Timezone difference to GMT+0: 3600s

Hourly data
                          date  temperature_2m  relative_humidity_2m  rain  \
0   2026-07-21 00:00:00+01:00       29.617001                  44.0   0.0   
1   2026-07-21 01:00:00+01:00       29.017000                  45.0   0.0   
2   2026-07-21 02:00:00+01:00       27.967001                  43.0   0.0   
3   2026-07-21 03:00:00+01:00       27.467001                  47.0   0.0   
4   2026-07-21 04:00:00+01:00       26.817001                  45.0   0.0   
..                        ...             ...                   ...   ...   
187 2026-07-28 19:00:00+01:00       27.017000                  73.0   0.0   
188 2026-07-28 20:00:00+01:00       25.667000                  83.0   0.0   
189 2026-07-28 21:00:00+01:00       25.167000                  86.0   0.0   
190 2026-07-28 22:00:00+01:00       25.667000                  77.0   0

In [38]:
# Setup the Open-Meteo API client with cache and retry on error
cache_session = requests_cache.CachedSession('.cache', expire_after = 3600)
retry_session = retry(cache_session, retries = 5, backoff_factor = 0.2)
openmeteo = openmeteo_requests.Client(session = retry_session)

# Make sure all required weather variables are listed here
# The order of variables in hourly or daily is important to assign them correctly below
url = "https://api.open-meteo.com/v1/forecast"

params = {
    "latitude": 43.2965,
    "longitude": 5.3698,
    "hourly": ["temperature_2m", "relative_humidity_2m", "rain", "surface_pressure", "wind_speed_10m"],
    "timezone": "Europe/London",
    "past_days": 7,
    "forecast_days": 1,
}
responses = openmeteo.weather_api(url, params = params)

# Process first location. Add a for-loop for multiple locations or weather models
response = responses[0]
print(f"Coordinates: {response.Latitude()}°N {response.Longitude()}°E")
print(f"Elevation: {response.Elevation()} m asl")
print(f"Timezone: {response.Timezone()}{response.TimezoneAbbreviation()}")
print(f"Timezone difference to GMT+0: {response.UtcOffsetSeconds()}s")

# Process hourly data. The order of variables needs to be the same as requested.
hourly = response.Hourly()
hourly_temperature_2m = hourly.Variables(0).ValuesAsNumpy()
hourly_relative_humidity_2m = hourly.Variables(1).ValuesAsNumpy()
hourly_rain = hourly.Variables(2).ValuesAsNumpy()
hourly_surface_pressure = hourly.Variables(3).ValuesAsNumpy()
hourly_wind_speed_10m = hourly.Variables(4).ValuesAsNumpy()

hourly_data = {
    "date": pd.date_range(
        start = pd.to_datetime(hourly.Time(), unit = "s", utc = True),
        end =  pd.to_datetime(hourly.TimeEnd(), unit = "s", utc = True),
        freq = pd.Timedelta(seconds = hourly.Interval()),
        inclusive = "left"
    ).tz_convert(response.Timezone().decode())
}

hourly_data["temperature_2m"] = hourly_temperature_2m
hourly_data["relative_humidity_2m"] = hourly_relative_humidity_2m
hourly_data["rain"] = hourly_rain
hourly_data["surface_pressure"] = hourly_surface_pressure
hourly_data["wind_speed_10m"] = hourly_wind_speed_10m

hourly_dataframe = pd.DataFrame(data = hourly_data)

Coordinates: 43.29999923706055°N 5.369998931884766°E
Elevation: 6.0 m asl
Timezone: b'Europe/London'b'GMT+1'
Timezone difference to GMT+0: 3600s


In [23]:
type(hourly_dataframe["date"].iloc[-1])

pandas._libs.tslibs.timestamps.Timestamp

In [35]:
hourly_dataframe

,date,temperature_2m,relative_humidity_2m,rain,surface_pressure,wind_speed_10m
0,2026-07-21 00:00:00+01:00,29.617001,44.0,0.0,1012.814331,1.835647
1,2026-07-21 01:00:00+01:00,29.017000,45.0,0.0,1013.012390,3.075841
2,2026-07-21 02:00:00+01:00,27.967001,43.0,0.0,1012.512024,8.089993
3,2026-07-21 03:00:00+01:00,27.467001,47.0,0.0,1012.710327,2.902413
4,2026-07-21 04:00:00+01:00,26.817001,45.0,0.0,1012.708374,3.319036
...,...,...,...,...,...,...
187,2026-07-28 19:00:00+01:00,27.017000,73.0,0.0,1017.605408,13.138765
188,2026-07-28 20:00:00+01:00,25.667000,83.0,0.0,1018.001587,11.570515
189,2026-07-28 21:00:00+01:00,25.167000,86.0,0.0,1018.599976,9.504273
190,2026-07-28 22:00:00+01:00,25.667000,77.0,0.0,1019.000916,8.145870


In [34]:
hourly_dataframe["date"].iloc[-1]

Timestamp('2026-07-28 23:00:00+0100', tz='Europe/London')

In [26]:
hourly_dataframe["date"].iloc[-1].to_pydatetime()

datetime.datetime(2026, 7, 28, 23, 0, tzinfo=<DstTzInfo 'Europe/London' BST+1:00:00 DST>)